# Grad-CAM Explainability Analysis

This notebook demonstrates how to use Grad-CAM++ to visualize which leaf regions
the model focuses on when making a disease prediction.

Contents:
1. Load trained model
2. Single image Grad-CAM visualization
3. Multi-class comparison (same leaf, different target classes)
4. Worst-confidence predictions — qualitative analysis
5. Attention region statistics

In [ ]:
import sys
sys.path.insert(0, '../..')

import asyncio
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

from backend.app.ml.models.model_manager import (
    EfficientNetB4PlantDisease, DISEASE_CLASSES, NUM_CLASSES
)
from backend.app.ml.preprocessing.image_preprocessor import ImagePreprocessor
from backend.app.ml.explainability.gradcam import GradCAMPlusPlus
from backend.app.core.config import settings

CHECKPOINT = Path('../../models/efficientnet_b4_best.pth')
DATA_DIR   = Path('../data/processed/plantvillage')
device     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# Load model
model = EfficientNetB4PlantDisease(NUM_CLASSES, pretrained=False)
if CHECKPOINT.exists():
    state = torch.load(CHECKPOINT, map_location=device, weights_only=True)
    if all(k.startswith('module.') for k in state):
        state = {k.removeprefix('module.'): v for k, v in state.items()}
    model.load_state_dict(state)
    print('Checkpoint loaded ✓')
else:
    print(f'WARNING: Checkpoint not found at {CHECKPOINT}. Using random weights.')

model = model.to(device).eval()
preprocessor = ImagePreprocessor(target_size=(380, 380))

## Single Image Grad-CAM

In [ ]:
TARGET_CLASS = 'tomato___early_blight'
sample_images = list((DATA_DIR / TARGET_CLASS).glob('*.jpg'))[:3]

fig, axes = plt.subplots(len(sample_images), 4, figsize=(16, 5 * len(sample_images)))
if len(sample_images) == 1:
    axes = [axes]

for row, img_path in enumerate(sample_images):
    img_bytes = img_path.read_bytes()

    # Preprocess
    prep = asyncio.get_event_loop().run_until_complete(
        preprocessor.preprocess_for_inference(img_bytes)
    )
    tensor = prep.tensor.to(device).requires_grad_(True)

    # Predict
    with torch.no_grad():
        logits = model(prep.tensor.to(device))
        probs  = torch.softmax(logits, -1).squeeze()

    pred_idx  = probs.argmax().item()
    pred_name = DISEASE_CLASSES[pred_idx]
    pred_conf = probs[pred_idx].item()

    # Grad-CAM
    gradcam = GradCAMPlusPlus(model, 'features.8')
    result  = gradcam.compute(tensor, class_idx=pred_idx, original_image=prep.original_image)
    gradcam.remove_hooks()

    # Plot
    ax = axes[row]
    ax[0].imshow(Image.open(img_path).resize((380, 380)))
    ax[0].set_title('Original', fontweight='bold')
    ax[0].axis('off')

    ax[1].imshow(result.heatmap, cmap='jet', vmin=0, vmax=1)
    ax[1].set_title('Grad-CAM++ Heatmap')
    ax[1].axis('off')

    ax[2].imshow(cv2.cvtColor(result.overlay, cv2.COLOR_BGR2RGB))
    ax[2].set_title(f'Overlay\n{pred_name}\n{pred_conf:.1%}')
    ax[2].axis('off')

    # Top-5 bar chart
    top5_idx  = probs.cpu().topk(5).indices.tolist()
    top5_prob = probs.cpu().topk(5).values.tolist()
    top5_name = [DISEASE_CLASSES[i].split('___')[1].replace('_', ' ').title()[:20] for i in top5_idx]
    ax[3].barh(top5_name[::-1], top5_prob[::-1], color='steelblue', alpha=0.8)
    ax[3].set_xlabel('Confidence')
    ax[3].set_title('Top-5 Predictions')
    ax[3].set_xlim(0, 1)

plt.suptitle(f'Grad-CAM Analysis: {TARGET_CLASS}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/gradcam_analysis.png', dpi=130, bbox_inches='tight')
plt.show()

## Multi-Class Comparison (Same Image, Different Targets)

In [ ]:
img_path = next((DATA_DIR / 'tomato___early_blight').glob('*.jpg'))
img_bytes = img_path.read_bytes()
prep = asyncio.get_event_loop().run_until_complete(
    preprocessor.preprocess_for_inference(img_bytes)
)

TARGET_CLASSES = [
    'tomato___early_blight',
    'tomato___late_blight',
    'tomato___healthy',
    'tomato___target_spot',
]

fig, axes = plt.subplots(1, len(TARGET_CLASSES) + 1, figsize=(4 * (len(TARGET_CLASSES) + 1), 4))

axes[0].imshow(Image.open(img_path).resize((224, 224)))
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

for col, cls in enumerate(TARGET_CLASSES, start=1):
    cls_idx = DISEASE_CLASSES.index(cls)
    tensor = prep.tensor.to(device).requires_grad_(True)

    gradcam = GradCAMPlusPlus(model, 'features.8')
    result = gradcam.compute(tensor, class_idx=cls_idx, original_image=prep.original_image)
    gradcam.remove_hooks()

    with torch.no_grad():
        logits = model(prep.tensor.to(device))
        prob = torch.softmax(logits, -1).squeeze()[cls_idx].item()

    axes[col].imshow(cv2.cvtColor(result.overlay, cv2.COLOR_BGR2RGB))
    axes[col].set_title(f'{cls.split("___")[1].replace("_", " ").title()}\n(conf={prob:.1%})', fontsize=9)
    axes[col].axis('off')

plt.suptitle('Grad-CAM for Different Target Classes (Same Image)', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/gradcam_multiclass.png', dpi=130, bbox_inches='tight')
plt.show()